<a href="https://colab.research.google.com/github/roly-3618/114-2_coding/blob/main/%E3%80%8CHW1_2_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Gradio_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1f0jkB_z5XxHhVhpB2PVDENNQ_PUrb-OSHE7CGpJg6G8/edit?gid=0#gid=0

In [5]:
import gradio as gr
import pandas as pd
import datetime
from datetime import timedelta, timezone
import gspread
import datetime
import plotly.express as px
from google.colab import auth
from google.auth import default

In [6]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1f0jkB_z5XxHhVhpB2PVDENNQ_PUrb-OSHE7CGpJg6G8/edit?gid=0#gid=0"
WORKSHEET_NAME = "工作表2"

REQUIRED_COLUMNS = ["日期", "時間", "分類", "品項", "金額", "付款人", "地點"]

_auth_done = False
_gc = None
_ws = None

In [7]:
# ... (前面的 _ensure_auth_and_ws, _ensure_headers, _read_df 等函數保持不變)

def add_expense(date_str, time_str, category, item, amount, payer, location, note):
    try:
        tz = timezone(timedelta(hours=8))
        now = datetime.datetime.now(tz)
        # 日期驗證
        if not date_str:
            date_str = now.strftime("%Y-%m-%d")

        # 時間驗證
        time_str = (time_str or "").strip()
        if not time_str:
            time_str = now.strftime("%H:%M")

        # 類別/品項/付款人預設
        category = (category or "未填").strip()
        item = (item or "未填").strip()
        payer = (payer or "匿名").strip()

        # 允許空白備註
        note = (note or "").strip()

        # 金額
        try:
            amount_val = float(amount)
        except:
            return "金額格式錯誤，請輸入數字", None, None, None

        ws = _ensure_auth_and_ws()
        # 直接 append 一列
        ws.append_row(
            [date_str, time_str, category, item, amount_val, payer, location, note],
            value_input_option="USER_ENTERED"
        )
        # 這裡的 Emoji 訊息也同步更新
        msg = f"✨ 已記錄：{date_str} {time_str}｜{category}｜{item}｜{amount_val}｜{payer}｜{location}｜{note}"
        # 回傳即時摘要
        df = _read_df()
        cat, settle = _make_summary_tables(df)
        total = float(df["金額"].sum()) if not df.empty else 0.0
        return msg, total, cat, settle
    except Exception as e:
        return f"⚠️ 新增失敗：{e}", None, None, None

# ... (refresh_summary, _make_summary_tables, _view_all_with_chart 保持不變)

with gr.Blocks(title="生活開支記帳本") as demo:
    gr.Markdown("## 📑 生活開支記帳本（Gradio）\n- 新增支出後自動寫回 Google Sheet\n- 一鍵查看總額、分類小計與 AA 分攤\n- 讀寫工作表：`工作表2`")
    tz = timezone(timedelta(hours=8))
    now_tw = datetime.datetime.now(tz)

    with gr.Tab("📝 新增支出"):
        # 快捷按鈕：更換品項
        gr.Markdown("### 🚀 快速填入")
        with gr.Row():
            btn_drink = gr.Button("🧋 手搖飲 $55")
            btn_noodle = gr.Button("🍜 牛肉麵 $180")
            btn_mrt = gr.Button("🚇 捷運 $35")

        # 輸入欄位
        with gr.Row():
            date_in = gr.Textbox(label="日期", value=now_tw.strftime("%Y-%m-%d"))
            time_in = gr.Textbox(label="時間", value=now_tw.strftime("%H:%M"), placeholder="留白自動偵測")
        with gr.Row():
            cat_in = gr.Textbox(label="分類", placeholder="如 餐飲 / 交通 / 娛樂")
            item_in = gr.Textbox(label="品項", placeholder="如 珍奶 / 晚餐 / 悠遊卡")
        with gr.Row():
            amt_in = gr.Textbox(label="金額", placeholder="數字")
            payer_in = gr.Textbox(label="付款人", placeholder="如 Pecu / Alice / Bob")
        with gr.Row():
            loc_in = gr.Textbox(label="地點", placeholder="如 50嵐 / 台北車站")
            note_in = gr.Textbox(label="備註", placeholder="如 幫同事代購 / 慶生")
        add_btn = gr.Button("確認新增")

        # 定義快捷按鈕邏輯
        def quick_fill(item, amt, cat):
            return cat, item, str(amt)

        # 綁定更新後的品項
        btn_drink.click(fn=lambda: quick_fill("手搖飲", 55, "餐飲"), outputs=[cat_in, item_in, amt_in])
        btn_noodle.click(fn=lambda: quick_fill("牛肉麵", 180, "餐飲"), outputs=[cat_in, item_in, amt_in])
        btn_mrt.click(fn=lambda: quick_fill("捷運", 35, "交通"), outputs=[cat_in, item_in, amt_in])

        add_msg = gr.Markdown()
        total_out = gr.Number(label="目前總額", interactive=False)
        cat_df = gr.Dataframe(label="分類小計", interactive=False)
        settle_df = gr.Dataframe(label="AA 分攤結算", interactive=False)

        add_btn.click(
            fn=add_expense,
            inputs=[date_in, time_in, cat_in, item_in, amt_in, payer_in, loc_in, note_in],
            outputs=[add_msg, total_out, cat_df, settle_df]
        )

    with gr.Tab("📈 彙總 / AA 分攤"):
        with gr.Row():
            # 設定每月預算
            budget_limit = gr.Number(label="💎 每月預算目標", value=15000)
            refresh_btn = gr.Button("🔄 刷新彙總數據", variant="primary")

        # 智慧進度條放在最上方
        budget_progress_html = gr.HTML()

        msg2 = gr.Markdown()

        with gr.Row():
            total2 = gr.Number(label="本月總支出", interactive=False)

        with gr.Row():
            cat_df2 = gr.Dataframe(label="分類小計", interactive=False)
            settle_df2 = gr.Dataframe(label="AA 分攤結算", interactive=False)

        raw_preview = gr.Dataframe(label="（預覽）最近資料", interactive=False)

        # 綁定點擊事件
        refresh_btn.click(
            fn=refresh_summary,
            inputs=[budget_limit],
            outputs=[msg2, total2, cat_df2, settle_df2, raw_preview, budget_progress_html]
        )

    with gr.Tab("🔍 檢視原始資料"):
        view_btn = gr.Button("📊 產生圖表與分析", variant="primary")

        # 新增 Plot 元件放圓餅圖
        view_chart = gr.Plot(label="支出佔比圖")

        # 表格元件
        view_df = gr.Dataframe(
            label="完整帳目清單",
            interactive=False,
            wrap=True
        )

        # 綁定點擊事件
        view_btn.click(
            fn=_view_all_with_chart,
            inputs=[],
            outputs=[view_chart, view_df]
        )

# ... (get_budget_progress 保持不變)

# 啟動介面
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://803ed09ec1d1ad54a6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
